# kBET / iLISI: metacell-composition vs. standard per-cell embedding — scProto vs. scPoli (Stage-1) vs. SEACells

**Reviewer concern this answers** (e9Ho, second-round comment — see `../reviews/reviews.txt:388`, `../my-notes/comments1/rev3.txt`):

> "On the standard integration metrics I take your point about NMI and ARI; measures such as kBET or iLISI would sidestep that objection while still giving a common point of comparison."

Our original objection (`../rebuttle-responses/reviewer3_e9Ho_full.md`, Weakness 3) was specifically about NMI/ARI: they require re-clustering the embedding into k = #annotated cell types and scoring agreement with those labels, which forces scProto's deliberately finer prototype structure (220–300 prototypes vs. 9–17 annotated types) to collapse into coarse bins — penalizing exactly the fine-grained structure scProto is built to preserve. kBET and iLISI don't have that flaw: both test **local neighborhood batch composition** directly, with no clustering-to-labels step at all.

**Two-tier design in this notebook:**

- **Part A — metacell-composition variant.** Standard kBET/iLISI define "neighborhood" as a cell's k-nearest-neighbors in a shared embedding. SEACells has no such embedding of its own — it only outputs a hard group (metacell) assignment — so there's no principled shared space to build a k-NN graph on for a SEACells row. Using **each method's own metacell/prototype assignment as the neighborhood** sidesteps this: it directly tests whether the actual groups a method outputs are batch-mixed, works uniformly across scProto / SEACells (PCA) / SEACells (scPoli Stage-1) / Leiden (scPoli Stage-1), and has no free k hyperparameter (the neighborhood size is exactly each metacell's own realized size). This is the modification discussed in chat: reuse the metacell assignment instead of picking a k-NN k.
  - `calc_metacell_ilisi` (`interpretable_ssl/evaluation/mc_metric_utils.py`) — inverse Simpson index of batch composition per metacell, same formula as iLISI, evaluated on metacell membership instead of a k-NN neighborhood.
  - `calc_metacell_kbet` (same file) — chi-squared test of each metacell's batch counts against the dataset's global batch proportions; reports the rejection rate (fraction of metacells that deviate significantly), same statistic kBET reports, again substituting metacell membership for a k-NN neighborhood.
  - Caveat, stated up front: this is **our own variant**, not literally the metric a reviewer would recognize from a scib-metrics scorecard — worth flagging as such in the response rather than calling it "kBET" unqualified.

- **Part B — standard per-cell embedding kBET/iLISI**, via `scib_metrics.benchmark.Benchmarker` (already wrapped as `get_scib` / `get_all_embeddings_for_scib` in `interpretable_ssl/evaluation/metric_helpers/embedding_metrics.py`), restricted to just kBET + iLISI (NMI/ARI/silhouette-label/isolated-labels turned off — see `_safe_metric_configs()` below) so we don't pay for the slow Leiden/KMeans clustering search just to discard the columns we're not using. Run on three embeddings: `X_pca` (raw, uncorrected — the space SEACells actually builds its kernel on, `build_kernel_on='X_pca'` is its default; the closest well-defined SEACells-equivalent row for a per-cell embedding test, since SEACells itself has no corrected embedding of its own), `X_stage1z` (scPoli Stage-1), and `X_scproto` (scProto's own Stage-2 latent). `get_all_embeddings_for_scib` already computes `X_pca` (50 comps) as part of its existing pipeline, so this is not new compute. This is the literal, standard computation, giving the reviewer the recognizable "common point of comparison" number, now including an uncorrected-PCA row as a lower-bound reference and SEACells proxy.

**Reading the two parts together:** Part A is the fair three/four-way comparison of what each method's actual output (its groups) does; Part B is the reviewer's literal ask, computed per-cell on the underlying continuous embeddings — including the one SEACells itself clusters on. Neither part touches NMI/ARI or a forced cell-type re-clustering. Important distinction to keep straight when reading Part B's `X_pca` row: it measures whether the *ambient, uncorrected* space is batch-mixed, not whether SEACells' own groups are — that's still Part A's job (the `SEACells (PCA)` row there).

**Scope:** all 3 RNA-seq datasets (pancreas, lung, pbmc-immune), matching every other rebuttal table. No retraining — Part A reads each method's already-saved `cell_assignments.csv` (ALL cells, no subsampling); Part B reloads existing scPoli-Stage1 / scProto checkpoints to encode (same reload `get_all_embeddings_for_scib` already does for the E9 scIB comparison).

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

In [ ]:
# IMPORTANT: restart the runtime after the cell above before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [ ]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test. (Same check as heldout_edge_modularity.ipynb / batch_correct_then_cluster_baselines.ipynb.)
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

In [ ]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

In [ ]:
# Extra imports not already covered by nb_setup.py.
import os
import dataclasses
import numpy as np
import pandas as pd
from scipy.stats import chisquare

from interpretable_ssl.evaluation.mc_metric_utils import calc_metacell_ilisi, calc_metacell_kbet
from interpretable_ssl.evaluation.metric_helpers.embedding_metrics import get_scib, get_all_embeddings_for_scib
from interpretable_ssl.evaluation.paper_figures import _resolve_run_dir
from interpretable_ssl.evaluation.rebuttal_report import build_model_keywords, SCPROTO_KEY
from interpretable_ssl.datasets.dataset_configs import DATASETS

print("extra imports ready")

## Config

In [ ]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

ALPHA = 0.05          # kBET rejection threshold
MIN_MC_SIZE = 5        # metacells smaller than this are excluded from the kBET test (unreliable chi-squared)

# Part A rows: scProto, SEACells (PCA, uncorrected), SEACells (scPoli Stage-1), Leiden (scPoli Stage-1).
# Reuses the SAME on-disk folder-tag convention every other rebuttal table in this repo uses
# (batch_correct_then_cluster_baselines.ipynb / heldout_edge_modularity.ipynb / rebuttal_report.py).
MODEL_KEYWORDS = build_model_keywords(
    correction_methods=['stage1z'],
    method_display_names={'stage1z': 'scPoli (Stage-1)'},
)
MODEL_KEYWORDS

## Part A — metacell-composition kBET / iLISI

For each dataset, load every method's `cell_assignments.csv` (all cells, no subsampling), compute a
single global batch-frequency reference from the full cell population, then score every method's
metacell assignment against that SAME reference — so a difference in rejection rate / iLISI reflects
the method's mixing, not a different reference distribution.

In [ ]:
def load_assignments_for_method(ds_id, keyword):
    """Read a method's per-cell assignment (metacell_id + label_key + batch_key,
    plus cell_id where available) via the same _resolve_run_dir lookup every
    other rebuttal table in this repo uses -- no retraining, no model loading.

    Tries cell_assignments.csv first (all cells, has cell_id). Falls back to
    umap_cells.csv in the SAME run dir if that's missing -- same underlying
    per-cell data, written by the same _save_seacell_umap_data / save_umap_data
    call, just without a cell_id column (dropped, not needed here) and with two
    extra umap_1/umap_2 columns (also dropped). Confirmed on disk: the plain
    'seacell' (SEACells (PCA)) run predates cell_assignments.csv being written
    at all, but its umap_cells.csv IS the full, unsubsampled population with
    exactly the columns needed (verified: row count matches n_metacells ==
    DATASETS[ds_id]['num_prototypes'] for all 3 RNA-seq datasets).

    Returns None if neither file is found.
    """
    run_dir = _resolve_run_dir(ds_id, keyword, prefer_csv='cell_assignments.csv')
    if run_dir is None:
        return None
    for fname in ('cell_assignments.csv', 'umap_cells.csv'):
        path = os.path.join(run_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path, usecols=lambda c: c not in ('umap_1', 'umap_2'))
            if fname == 'umap_cells.csv':
                print(f"[{ds_id}] {keyword}: cell_assignments.csv missing -- "
                      f"using umap_cells.csv instead ({len(df)} cells).")
            return df
    return None


def weighted_mean_by_mc_size(per_mc_series, mc_sizes):
    """per_mc_series is str-indexed (calc_metacell_ilisi's convention);
    mc_sizes (from value_counts()) may be int-indexed -- align by string."""
    sizes = mc_sizes.copy()
    sizes.index = sizes.index.astype(str)
    w = sizes.reindex(per_mc_series.index).fillna(0)
    if w.sum() == 0:
        return None
    return float((per_mc_series * w).sum() / w.sum())

In [ ]:
metacell_rows = []

# Stashed per-metacell series (not just the mean) for every (dataset, method) --
# reused by the significance-test cell below instead of recomputing.
ilisi_per_mc_store = {}   # ilisi_per_mc_store[ds_id][method_disp] -> pd.Series
kbet_per_mc_store = {}    # kbet_per_mc_store[ds_id][method_disp]  -> pd.DataFrame (p_value, rejected, size)

for ds_id in RNA_SEQ_DATASETS:
    bk = DATASETS[ds_id].get('batch_key')
    if bk is None:
        print(f"[{ds_id}] no batch_key configured -- skipping (single-batch dataset).")
        continue

    dfs = {}
    for keyword, disp in MODEL_KEYWORDS.items():
        df = load_assignments_for_method(ds_id, keyword)
        if df is None:
            print(f"[{ds_id}] {disp} ({keyword}): cell_assignments.csv not found -- skipping.")
            continue
        if bk not in df.columns:
            print(f"[{ds_id}] {disp}: batch key '{bk}' not in cell_assignments.csv -- skipping.")
            continue
        dfs[disp] = df

    if not dfs:
        print(f"[{ds_id}] no methods available -- skipping dataset entirely.")
        continue

    # Global batch frequency: computed once per dataset, from whichever method's file
    # loaded first -- every method's cell_assignments.csv covers the same full cell
    # population, so any of them gives the same reference distribution. Cross-checked
    # against every other loaded method's row count below.
    ref_disp, ref_df = next(iter(dfs.items()))
    global_freq = ref_df[bk].value_counts(normalize=True)
    n_ref = len(ref_df)
    for disp, df in dfs.items():
        if len(df) != n_ref:
            print(f"[{ds_id}] WARNING: {disp} has {len(df)} cells, {ref_disp} (reference) has "
                  f"{n_ref} -- populations don't match; global_freq reference may not be fair "
                  f"for this row.")

    ilisi_per_mc_store[ds_id] = {}
    kbet_per_mc_store[ds_id] = {}

    for disp, df in dfs.items():
        ilisi_per_mc = calc_metacell_ilisi(df, batch_key=bk, mc_key='metacell_id', return_per_mc=True)
        mc_sizes = df['metacell_id'].value_counts()
        ilisi_weighted = weighted_mean_by_mc_size(ilisi_per_mc, mc_sizes)

        kbet_res = calc_metacell_kbet(
            df, batch_key=bk, mc_key='metacell_id',
            global_freq=global_freq, alpha=ALPHA, min_size=MIN_MC_SIZE, return_per_mc=True,
        )
        ilisi_per_mc_store[ds_id][disp] = ilisi_per_mc
        kbet_per_mc_store[ds_id][disp] = kbet_res.get('per_mc') if kbet_res else None

        metacell_rows.append({
            'dataset': ds_id,
            'method': disp,
            'n_metacells': int(df['metacell_id'].nunique()),
            'metacell_ilisi_mean': float(ilisi_per_mc.mean()) if ilisi_per_mc is not None else None,
            'metacell_ilisi_weighted_mean': ilisi_weighted,
            'kbet_rejection_rate_by_mc': kbet_res['rejection_rate_by_mc'] if kbet_res else None,
            'kbet_rejection_rate_by_cell': kbet_res['rejection_rate_by_cell'] if kbet_res else None,
            'kbet_n_tested': kbet_res['n_tested'] if kbet_res else None,
            'kbet_n_excluded_small': kbet_res['n_excluded_small'] if kbet_res else None,
        })

df_metacell = pd.DataFrame(metacell_rows).set_index(['dataset', 'method'])
df_metacell.round(3)

In [ ]:
pd.set_option('display.width', 160)
summary_a = df_metacell.rename(index=dataset_display_names, level='dataset')[
    ['n_metacells', 'metacell_ilisi_mean', 'metacell_ilisi_weighted_mean',
     'kbet_rejection_rate_by_mc', 'kbet_rejection_rate_by_cell', 'kbet_n_excluded_small']
].round(3)
summary_a

### Significance test: scProto vs. SEACells (expect better) / vs. scPoli-Stage-1 methods (expect not worse)

Unpaired one-sided Mann-Whitney U on the per-metacell iLISI / kBET-p-value distributions --
each metacell is one sample, scProto's metacells vs. the comparison method's metacells (different
n_metacells per method, so unpaired rather than the paired-per-batch tests used elsewhere in this
rebuttal). Higher iLISI = better mixing; higher kBET p-value = less rejected = better mixing -- same
"higher is better" orientation for both, so one test function covers both metrics.

Two different one-sided hypotheses, matching the two different expectations:
- **vs. SEACells (PCA)**: `alternative='greater'` -- testing whether scProto's distribution is
  stochastically GREATER (significant + scProto ahead = expectation confirmed).
- **vs. SEACells (scPoli Stage-1) / Leiden (scPoli Stage-1)**: `alternative='less'` -- testing
  whether scProto's distribution is stochastically LESS. This is the hypothesis that would
  *contradict* "almost similar, not worse" -- non-significant here supports the expectation,
  significant here contradicts it. (Note: non-significant is NOT proof of equivalence, just
  insufficient evidence of a difference -- see caveat printed below the table.)

Bonferroni correction: 3 comparisons per dataset (vs. each of the other 3 methods).

In [ ]:
from scipy.stats import mannwhitneyu

def mwu_one_sided(sample_scproto, sample_other, alternative):
    a = pd.Series(sample_scproto).dropna().values
    b = pd.Series(sample_other).dropna().values
    if len(a) < 2 or len(b) < 2:
        return {'u_stat': None, 'p': None, 'n_scproto': len(a), 'n_other': len(b)}
    u, p = mannwhitneyu(a, b, alternative=alternative)
    return {'u_stat': float(u), 'p': float(p), 'n_scproto': len(a), 'n_other': len(b)}


# hypothesis per comparison method: 'greater' = testing scProto > other (expect significant);
# 'less' = testing scProto < other (expect NOT significant, i.e. p_adj > 0.05)
COMPARISON_HYPOTHESIS = {
    'SEACells (PCA)': 'greater',
    'SEACells (scPoli (Stage-1))': 'less',
    'Leiden (scPoli (Stage-1))': 'less',
}
N_COMPARISONS_PER_DATASET = len(COMPARISON_HYPOTHESIS)

sig_rows = []
for ds_id in RNA_SEQ_DATASETS:
    if ds_id not in ilisi_per_mc_store or 'scProto' not in ilisi_per_mc_store[ds_id]:
        continue
    scproto_ilisi = ilisi_per_mc_store[ds_id]['scProto']
    scproto_kbet = kbet_per_mc_store[ds_id].get('scProto')
    scproto_kbet_p = scproto_kbet['p_value'] if scproto_kbet is not None else None

    for other_disp, alternative in COMPARISON_HYPOTHESIS.items():
        if other_disp not in ilisi_per_mc_store[ds_id]:
            continue

        ilisi_res = mwu_one_sided(scproto_ilisi, ilisi_per_mc_store[ds_id][other_disp], alternative)
        ilisi_res['p_adj'] = (
            min(1.0, ilisi_res['p'] * N_COMPARISONS_PER_DATASET) if ilisi_res['p'] is not None else None
        )

        other_kbet = kbet_per_mc_store[ds_id].get(other_disp)
        other_kbet_p = other_kbet['p_value'] if other_kbet is not None else None
        kbet_res = mwu_one_sided(scproto_kbet_p, other_kbet_p, alternative)
        kbet_res['p_adj'] = (
            min(1.0, kbet_res['p'] * N_COMPARISONS_PER_DATASET) if kbet_res['p'] is not None else None
        )

        sig_rows.append({
            'dataset': ds_id,
            'vs_method': other_disp,
            'hypothesis': f"scProto {'>' if alternative == 'greater' else '<'} {other_disp}",
            'ilisi_median_scproto': float(pd.Series(scproto_ilisi).median()),
            'ilisi_median_other': float(pd.Series(ilisi_per_mc_store[ds_id][other_disp]).median()),
            'ilisi_p_adj': ilisi_res['p_adj'],
            'ilisi_n_scproto': ilisi_res['n_scproto'],
            'ilisi_n_other': ilisi_res['n_other'],
            'kbet_p_median_scproto': float(pd.Series(scproto_kbet_p).median()) if scproto_kbet_p is not None else None,
            'kbet_p_median_other': float(pd.Series(other_kbet_p).median()) if other_kbet_p is not None else None,
            'kbet_p_adj': kbet_res['p_adj'],
        })

df_sig_a = pd.DataFrame(sig_rows)
df_sig_a['dataset'] = df_sig_a['dataset'].map(dataset_display_names)
pd.set_option('display.width', 200)
df_sig_a.round(4)

**Reading this table:** for the `SEACells (PCA)` rows, `ilisi_p_adj < 0.05` (with scProto's median
above SEACells') confirms "significantly better". For the `scPoli (Stage-1)` rows,
`ilisi_p_adj < 0.05` means the "scProto < other" hypothesis IS supported -- i.e. scProto is
significantly *worse*, contradicting "almost similar, not worse"; `ilisi_p_adj >= 0.05` means there
isn't significant evidence scProto is worse (consistent with, but not proof of, "almost similar").
kBET's columns use the same logic on p-values instead of iLISI values -- keep in mind Part A's kBET
rejection rate was already near-saturated (0.9-1.0) for every method, so this test is likely
underpowered to detect anything on kBET specifically; iLISI is the metric actually worth reading here.

## Part B — standard per-cell embedding kBET / iLISI

Reuses the existing `get_all_embeddings_for_scib` / `get_scib` wrappers around
`scib_metrics.benchmark.Benchmarker` (already used for the full scIB battery elsewhere in this
rebuttal), restricted to just kBET + iLISI via `BioConservation`/`BatchCorrection` flags so the slow
NMI/ARI clustering search isn't run just to discard the columns. `_safe_metric_configs()` verifies the
installed `scib-metrics` version accepts these field names before running anything, and falls back to
the full default battery (printing the actual dataclass fields) if not, so a version mismatch fails
loudly with a fix, not silently.

Includes `X_pca` (raw, uncorrected) alongside `X_stage1z` / `X_scproto` — `X_pca` is the space SEACells
itself builds its kernel on by default (`build_kernel_on='X_pca'`), so its per-cell kBET/iLISI is the
best available SEACells-equivalent row for this literal, embedding-based version of the test (SEACells
has no corrected embedding of its own to test directly).

In [ ]:
from scib_metrics.benchmark import BioConservation, BatchCorrection

def _safe_metric_configs():
    try:
        bio_off = BioConservation(
            isolated_labels=False, nmi_ari_cluster_labels_leiden=False,
            nmi_ari_cluster_labels_kmeans=False, silhouette_label=False, clisi_knn=False,
        )
        batch_kbet_ilisi_only = BatchCorrection(
            silhouette_batch=False, ilisi_knn=True, kbet_per_label=True,
            graph_connectivity=False, pcr_comparison=False,
        )
        return bio_off, batch_kbet_ilisi_only
    except TypeError as e:
        print(f"BioConservation/BatchCorrection field names don't match this scib_metrics "
              f"version ({e}). Falling back to the full default battery (slower -- includes "
              f"the NMI/ARI clustering search) and selecting kBET/iLISI columns afterward instead.")
        print("BioConservation fields:", [f.name for f in dataclasses.fields(BioConservation)])
        print("BatchCorrection fields:", [f.name for f in dataclasses.fields(BatchCorrection)])
        return None, None

BIO_OFF, BATCH_KBET_ILISI_ONLY = _safe_metric_configs()

In [ ]:
# X_pca: raw/uncorrected -- the space SEACells itself builds its kernel on by
# default (build_kernel_on='X_pca'), included as the closest available
# SEACells-equivalent row for this per-cell embedding test (see intro cell).
EMBEDDING_KEYS = ['X_pca', 'X_stage1z', 'X_scproto']
EMBEDDING_DISPLAY = {
    'X_pca': 'PCA (uncorrected, SEACells space)',
    'X_stage1z': 'scPoli (Stage-1)',
    'X_scproto': 'scProto (embedding)',
}

embedding_rows = []
for ds_id in RNA_SEQ_DATASETS:
    lk = DATASETS[ds_id]['label_key']
    bk = DATASETS[ds_id].get('batch_key')
    print(f"\n=== [{ds_id}] standard per-cell kBET/iLISI ===")

    ad_scib = get_all_embeddings_for_scib(ds_id)
    keys_present = [k for k in EMBEDDING_KEYS if k in ad_scib.obsm]
    missing_keys = [k for k in EMBEDDING_KEYS if k not in ad_scib.obsm]
    if missing_keys:
        print(f"[{ds_id}] proceeding without: {missing_keys}")
    if not keys_present:
        print(f"[{ds_id}] no embeddings available -- skipping dataset.")
        continue

    scib_df = get_scib(
        ad_scib, keys_present, bk, lk,
        bio_conservation_metrics=BIO_OFF, batch_correction_metrics=BATCH_KBET_ILISI_ONLY,
    )
    if scib_df is None:
        print(f"[{ds_id}] get_scib returned None (single batch?) -- skipping.")
        continue

    print(f"[{ds_id}] scib_df columns: {list(scib_df.columns)}")  # verify exact kBET/iLISI column names at runtime
    scib_df = scib_df.rename(index=EMBEDDING_DISPLAY)
    # scib_metrics' own result index is typically named 'Embedding' (or similar),
    # NOT 'index' -- reset_index() would then create a column with THAT name, so
    # renaming {'index': 'method'} silently no-ops and 'method' never appears.
    # Force the index name first so reset_index() always produces 'method'.
    scib_df.index.name = 'method'
    scib_df = scib_df.reset_index()
    scib_df.insert(0, 'dataset', ds_id)
    embedding_rows.append(scib_df)

df_embedding = pd.concat(embedding_rows, ignore_index=True) if embedding_rows else pd.DataFrame()
df_embedding

In [ ]:
# Column names/casing can vary slightly by scib-metrics version -- match by
# substring instead of hardcoding, and print what matched so it's easy to verify.
kbet_ilisi_cols = [c for c in df_embedding.columns if 'kbet' in c.lower() or 'lisi' in c.lower()]
print("Detected kBET/iLISI columns:", kbet_ilisi_cols)

summary_b = df_embedding.set_index(['dataset', 'method'])[kbet_ilisi_cols].round(3)
summary_b = summary_b.rename(index=dataset_display_names, level='dataset')
summary_b

## Not covered here

- **Graph connectivity / PCR comparison / silhouette-batch** (other scib-metrics batch-correction
  metrics) — turned off in Part B to keep this response narrow to exactly what e9Ho's follow-up asked
  for (kBET, iLISI); flip the relevant `BatchCorrection` flags on if a broader battery is wanted later.
- **SEACells' own groups in Part B** — `X_pca` in Part B tests the *ambient space* SEACells clusters
  on, not SEACells' own output; Part A's metacell-composition variant (the `SEACells (PCA)` row) is
  still the only row that scores what SEACells itself actually produces.
- **Statistical significance / cross-batch variance** for either table — these are single point
  estimates per (dataset, method), unlike the batch-level mean±std reported elsewhere in this rebuttal
  (`calc_modularity_per_batch`-style breakdown). Add if a reviewer specifically asks for it.